In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\antek\Customer_Chunrn_End_to_End")

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Project source not found: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", Path.cwd())
print("Python:", sys.executable)

Working directory: C:\Users\antek\Customer_Chunrn_End_to_End
Python: c:\Users\antek\Customer_Chunrn_End_to_End\.venv\Scripts\python.exe


In [3]:
from pathlib import Path

from src.data.dashboard import open_portfolio
from src.models.explain import ChurnExplainer, waterfall_figure
from src.models.pipeline import FEATURE_COLUMNS
from src.models.predict import ChurnPredictor

predictor = ChurnPredictor.from_final_test(
    Path("mlruns"),
    final_test_run_id="5818be9683af42a485ffc2896a868137",
)
explainer = ChurnExplainer(predictor)

with open_portfolio(Path(r"data\processed\portfolio_v1")) as connection:
    training = connection.sql(
        "SELECT * FROM portfolio WHERE partition = 'train'"
    ).df()

inputs = training.loc[:, ["customer_id", *FEATURE_COLUMNS]]

# Local explanation
local = explainer.explain_one(inputs.iloc[0].to_dict())
display(local.contributions)
waterfall_figure(local, top_n=10).show()

# Global importance: deterministic, bounded training sample
global_result = explainer.global_importance(
    training.loc[:, ["customer_id", "partition", *FEATURE_COLUMNS]],
    sample_size=1000,
)
display(global_result.importance)

c:\Users\antek\Customer_Chunrn_End_to_End\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,feature,feature_value,shap_log_odds
0,age,43.0,-2.769220e-04
1,annual_income,55085.25,-4.862920e-04
2,dependents,1.0,-3.893649e-04
3,tenure,2.0,3.514578e-02
4,monthlycharges,67.2,-1.402507e-02
5,totalcharges,144.39,3.868259e-03
6,num_services,1.0,1.306459e-01
7,customer_satisfaction,9.0,-2.639618e-01
8,num_complaints,0.0,-1.360814e-01
9,num_service_calls,0.0,-1.712485e-01


,feature,mean_abs_shap_log_odds,mean_shap_log_odds,importance_share
0,contract,4.591957e-01,1.228640e-02,3.129214e-01
1,customer_satisfaction,2.179336e-01,8.042784e-03,1.485120e-01
2,has_tech_support,1.433818e-01,-8.659135e-03,9.770829e-02
3,num_complaints,1.291998e-01,1.669911e-03,8.804388e-02
4,num_service_calls,1.210252e-01,6.324446e-03,8.247328e-02
5,late_payments,1.178000e-01,-1.626817e-03,8.027546e-02
6,num_services,1.050068e-01,7.030107e-03,7.155747e-02
7,has_online_security,9.953292e-02,-2.273938e-03,6.782725e-02
8,monthlycharges,2.745851e-02,1.574442e-03,1.871175e-02
9,tenure,2.516164e-02,9.461969e-04,1.714654e-02
